# Voxer: Text-to-3D Voxel Generation

## Pipeline Overview

1. **Data Collection** - Download 3D models from Objaverse, convert to 64^3 voxels
2. **MAE Pretraining** - Mask and reconstruct voxels to learn 3D representations
3. **VQ-VAE Training** - Insert VQ bottleneck for discrete tokenization (voxel <-> token)
4. **GPT Training** - Text-conditioned autoregressive transformer generates token sequences
5. **Evaluation** - Generate 3D models from text descriptions

## Compute Requirements
- Stage 2 (MAE): ~50-100 GPU-hours on A100
- Stage 3 (VQ-VAE): ~30-60 GPU-hours
- Stage 4 (GPT): ~100-200 GPU-hours
- Total: ~200-400 GPU-hours

# 0. Environment Setup

In [ ]:
!pip install objaverse trimesh sentence-transformers tqdm matplotlib --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/voxer_checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/voxer_data', exist_ok=True)

In [ ]:
import sys
sys.path.insert(0, '/content')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm.auto import tqdm
import gc
import json
import os

from voxer.mae import VoxelMAE, mae_vit_base
from voxer.vqvae import VectorQuantizerEMA, VoxelVQVAE, create_vqvae_from_mae
from voxer.transformer import VoxelGPT
from voxer.data import PreprocessedVoxelDataset
from voxer.train import train_mae, train_vqvae, train_generator
from voxer.eval import evaluate_reconstruction, visualize_voxel, generate_and_visualize
from voxer.utils import set_seed

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Stage 1: Data Collection & Preprocessing

Download chairs (and optionally more categories) from Objaverse,
convert to 64x64x64 RGBA voxels, and compute text embeddings.

In [ ]:
# ---- Configuration ----
NUM_MODELS = 5000  # Increase for better results
VOXEL_RES = 64
BASE_DIR = '/content/hf_models'
os.makedirs(BASE_DIR, exist_ok=True)

# ---- Download annotations ----
import objaverse
print('Loading annotations...')
annotations = objaverse.load_annotations()

# ---- Filter models with rich descriptions ----
print('Filtering models with descriptions...')
valid_uids = []
category_keywords = ['chair', 'table', 'sofa', 'lamp', 'cabinet', 'desk', 'bed', 'shelf', 'stool', 'bench']

for uid, anno in annotations.items():
    name = str(anno.get('name', '')).lower()
    tags = ' '.join([str(t.get('name', '')).lower() for t in anno.get('tags', [])])
    desc = str(anno.get('description', ''))
    text = name + ' ' + tags
    if any(kw in text for kw in category_keywords) and len(desc) > 20:
        valid_uids.append(uid)

print(f'Found {len(valid_uids)} models with rich descriptions')

selected_uids = valid_uids[:NUM_MODELS]
del annotations; gc.collect()

In [ ]:
# ---- Download models ----
print(f'Downloading {len(selected_uids)} models...')
downloaded = objaverse.load_objects(
    uids=selected_uids,
    download_processes=8
)

import shutil
final_objects = {}
for uid, path in downloaded.items():
    final_path = os.path.join(BASE_DIR, f'{uid}.glb')
    if os.path.exists(path):
        shutil.copy(path, final_path)
        final_objects[uid] = final_path

print(f'Downloaded {len(final_objects)} models to {BASE_DIR}')

In [ ]:
# ---- Convert to Voxels ----
import trimesh

def model_to_voxel(file_path, resolution=VOXEL_RES):
    try:
        scene = trimesh.load(file_path)
        if isinstance(scene, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(scene.geometry.values()))
        else:
            mesh = scene
        points, face_idx = trimesh.sample.sample_surface(mesh, 100000)
        if hasattr(mesh.visual, 'material') and hasattr(mesh.visual.material, 'baseColorTexture'):
            texture = mesh.visual.material.baseColorTexture
            if texture is not None:
                texture = np.array(texture)
                uv = mesh.visual.uv
                faces = mesh.faces
                face_uv = uv[faces[face_idx]]
                bary = trimesh.triangles.points_to_barycentric(mesh.triangles[face_idx], points)
                sample_uv = (face_uv * bary[:, :, None]).sum(axis=1)
                H, W = texture.shape[:2]
                u, v = sample_uv[:, 0], 1.0 - sample_uv[:, 1]
                px = np.clip((u * (W - 1)).astype(int), 0, W - 1)
                py = np.clip((v * (H - 1)).astype(int), 0, H - 1)
                colors = texture[py, px][:, :3]
            else:
                colors = np.tile([128, 128, 128], (len(points), 1))
        else:
            colors = np.tile([128, 128, 128], (len(points), 1))
        mins, maxs = points.min(axis=0), points.max(axis=0)
        points_norm = (points - mins) / (maxs - mins + 1e-8)
        coords = (points_norm * (resolution - 1)).astype(int)
        voxel = np.zeros((resolution, resolution, resolution, 4), dtype=np.uint8)
        for c, color in zip(coords, colors):
            voxel[c[0], c[1], c[2], :3] = color
            voxel[c[0], c[1], c[2], 3] = 255
        return voxel
    except Exception as e:
        return None

print('Converting models to voxels...')
all_voxels = []
all_uids = []
model_files = [f for f in os.listdir(BASE_DIR) if f.endswith('.glb')]

for filename in tqdm(model_files):
    uid = filename.replace('.glb', '')
    file_path = os.path.join(BASE_DIR, filename)
    res = model_to_voxel(file_path)
    if res is not None:
        all_voxels.append(res)
        all_uids.append(uid)

dataset = np.array(all_voxels, dtype=np.uint8)
np.save('/content/voxels_dataset.npy', dataset)
print(f'Converted {len(dataset)} models. Shape: {dataset.shape}')

In [ ]:
# ---- Save descriptions and compute text embeddings ----
print('Loading annotations for descriptions...')
annotations = objaverse.load_annotations()
descriptions = {}
for uid in all_uids:
    if uid in annotations:
        desc = annotations[uid].get('description', '')
        name = annotations[uid].get('name', '')
        descriptions[uid] = f'{name}. {desc}'

with open('/content/descriptions.json', 'w', encoding='utf-8') as f:
    json.dump(descriptions, f, ensure_ascii=False, indent=2)

del annotations; gc.collect()

# ---- Compute embeddings with sentence-transformers ----
from sentence_transformers import SentenceTransformer
print('Computing text embeddings...')
text_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [descriptions.get(uid, 'a 3d model') for uid in all_uids]
embeddings = text_model.encode(texts, show_progress_bar=True, batch_size=64)
torch.save(torch.from_numpy(embeddings).float(), '/content/text_embeddings.pt')
print(f'Computed {len(embeddings)} text embeddings, dim={embeddings.shape[1]}')

print('\nStage 1 complete!')

# Stage 2: MAE Pretraining

Train 3D Masked Autoencoder to reconstruct masked voxels.
This learns rich 3D shape representations.

In [ ]:
# ---- MAE Configuration ----
MAE_CONFIG = {
    'voxel_size': 64,
    'patch_size': 8,
    'in_channels': 4,
    'embed_dim': 768,
    'encoder_depth': 12,
    'encoder_heads': 12,
    'decoder_depth': 4,
    'decoder_heads': 12,
    'decoder_embed_dim': 384,
    'mask_ratio': 0.75,
    'batch_size': 8,  # Adjust based on GPU memory
    'epochs': 200,
    'lr': 1.5e-4,
    'warmup_epochs': 10,
    'weight_decay': 0.05,
    'gradient_accumulation_steps': 2,
}

print(f'MAE Config:')
for k, v in MAE_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ---- Load preprocessed data ----
print('Loading voxel data...')
voxel_data = np.load('/content/voxels_dataset.npy')
print(f'Loaded {len(voxel_data)} samples, shape: {voxel_data.shape}')

dataset = PreprocessedVoxelDataset(voxel_data)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_ds, batch_size=MAE_CONFIG['batch_size'], shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=MAE_CONFIG['batch_size'], shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

In [ ]:
# ---- Create and Train MAE ----
mae_model = mae_vit_base(
    voxel_size=MAE_CONFIG['voxel_size'],
    patch_size=MAE_CONFIG['patch_size'],
    in_channels=MAE_CONFIG['in_channels'],
    mask_ratio=MAE_CONFIG['mask_ratio'],
)

n_params = sum(p.numel() for p in mae_model.parameters()) / 1e6
print(f'MAE Model parameters: {n_params:.2f}M')

mae_model = train_mae(
    model=mae_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=MAE_CONFIG['epochs'],
    lr=MAE_CONFIG['lr'],
    warmup_epochs=MAE_CONFIG['warmup_epochs'],
    weight_decay=MAE_CONFIG['weight_decay'],
    device=device,
    save_dir='/content/drive/MyDrive/voxer_checkpoints',
    mixed_precision=True,
    gradient_accumulation_steps=MAE_CONFIG['gradient_accumulation_steps'],
)

print('Stage 2 complete!')

# Stage 3: VQ-VAE Training

Insert VQ bottleneck into MAE to get discrete tokens.
voxel (64^3 x 4ch) -> MAE encoder -> [512 x 768] -> VQ (8192 codes) -> [512 tokens] -> MAE decoder -> voxel

In [ ]:
# ---- VQ-VAE Configuration ----
VQVAE_CONFIG = {
    'num_embeddings': 8192,
    'embedding_dim': 256,
    'commitment_cost': 0.25,
    'decay': 0.99,
    'batch_size': 8,
    'epochs': 100,
    'lr': 1e-4,
    'gradient_accumulation_steps': 2,
}

print(f'VQ-VAE Config:')
for k, v in VQVAE_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ---- Create VQ-VAE from pretrained MAE ----
vqvae_model = create_vqvae_from_mae(
    mae_model=mae_model,
    num_embeddings=VQVAE_CONFIG['num_embeddings'],
    embedding_dim=VQVAE_CONFIG['embedding_dim'],
    commitment_cost=VQVAE_CONFIG['commitment_cost'],
    decay=VQVAE_CONFIG['decay'],
)

n_params = sum(p.numel() for p in vqvae_model.parameters()) / 1e6
print(f'VQ-VAE parameters: {n_params:.2f}M')

In [ ]:
# ---- Recreate dataloaders for VQ-VAE (with same batch size) ----
train_loader_vqvae = DataLoader(
    train_ds, batch_size=VQVAE_CONFIG['batch_size'], shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader_vqvae = DataLoader(
    val_ds, batch_size=VQVAE_CONFIG['batch_size'], shuffle=False,
    num_workers=2, pin_memory=True
)

# ---- Train VQ-VAE ----
vqvae_model = train_vqvae(
    model=vqvae_model,
    train_loader=train_loader_vqvae,
    val_loader=val_loader_vqvae,
    epochs=VQVAE_CONFIG['epochs'],
    lr=VQVAE_CONFIG['lr'],
    device=device,
    save_dir='/content/drive/MyDrive/voxer_checkpoints',
    mixed_precision=True,
    gradient_accumulation_steps=VQVAE_CONFIG['gradient_accumulation_steps'],
)

print('Stage 3 complete!')

# Stage 4: Text-Conditioned Token Generation

Train GPT-style transformer to generate token sequences from text embeddings.
Text -> LLM embedding -> Transformer -> token sequence -> VQ-VAE decoder -> 3D voxel

In [ ]:
# ---- Generator Configuration ----
GPT_CONFIG = {
    'vocab_size': VQVAE_CONFIG['num_embeddings'],
    'num_tokens': 512,  # 8x8x8 grid
    'text_embed_dim': 384,  # all-MiniLM-L6-v2 output dim
    'hidden_dim': 768,
    'num_layers': 12,
    'num_heads': 12,
    'dropout': 0.1,
    'batch_size': 16,
    'epochs': 300,
    'lr': 3e-4,
    'warmup_epochs': 10,
    'weight_decay': 0.1,
    'gradient_accumulation_steps': 1,
}

print(f'GPT Config:')
for k, v in GPT_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ---- Load text embeddings and create dataset ----
text_embeddings = torch.load('/content/text_embeddings.pt')
print(f'Loaded text embeddings: {text_embeddings.shape}')

ds_with_text = PreprocessedVoxelDataset(voxel_data, text_embeddings.numpy())

gpt_train_size = int(0.8 * len(ds_with_text))
gpt_val_size = int(0.1 * len(ds_with_text))
gpt_test_size = len(ds_with_text) - gpt_train_size - gpt_val_size
gpt_train_ds, gpt_val_ds, gpt_test_ds = torch.utils.data.random_split(
    ds_with_text, [gpt_train_size, gpt_val_size, gpt_test_size],
    generator=torch.Generator().manual_seed(42)
)

gpt_train_loader = DataLoader(
    gpt_train_ds, batch_size=GPT_CONFIG['batch_size'], shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
gpt_val_loader = DataLoader(
    gpt_val_ds, batch_size=GPT_CONFIG['batch_size'], shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train: {len(gpt_train_ds)}, Val: {len(gpt_val_ds)}, Test: {len(gpt_test_ds)}')

In [ ]:
# ---- Create Generator ----
gpt_model = VoxelGPT(
    vocab_size=GPT_CONFIG['vocab_size'],
    num_tokens=GPT_CONFIG['num_tokens'],
    text_embed_dim=GPT_CONFIG['text_embed_dim'],
    hidden_dim=GPT_CONFIG['hidden_dim'],
    num_layers=GPT_CONFIG['num_layers'],
    num_heads=GPT_CONFIG['num_heads'],
    dropout=GPT_CONFIG['dropout'],
)

n_params = sum(p.numel() for p in gpt_model.parameters()) / 1e6
print(f'Generator parameters: {n_params:.2f}M')

In [ ]:
# ---- Train Generator ----
gpt_model = train_generator(
    model=gpt_model,
    vqvae=vqvae_model,
    train_loader=gpt_train_loader,
    val_loader=gpt_val_loader,
    epochs=GPT_CONFIG['epochs'],
    lr=GPT_CONFIG['lr'],
    warmup_epochs=GPT_CONFIG['warmup_epochs'],
    weight_decay=GPT_CONFIG['weight_decay'],
    device=device,
    save_dir='/content/drive/MyDrive/voxer_checkpoints',
    mixed_precision=True,
    gradient_accumulation_steps=GPT_CONFIG['gradient_accumulation_steps'],
)

print('Stage 4 complete!')

# Stage 5: Evaluation & Generation

Evaluate reconstruction quality and generate 3D models from text.

In [ ]:
# ---- Evaluate VQ-VAE Reconstruction ----
vqvae_model.eval()
test_loader = DataLoader(
    test_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True
)

results = evaluate_reconstruction(vqvae_model, test_loader, device, max_batches=20)
print('\n===== Reconstruction Results =====')
print(f'MSE: {results["mse"]:.4f}')
print(f'IoU: {results["iou"]:.4f}')
if 'codebook_usage' in results:
    print(f'Codebook Usage: {results["codebook_usage"]:.2%}')

In [ ]:
# ---- Generate 3D Models from Text ----
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('all-MiniLM-L6-v2')

test_texts = [
    'a wooden dining chair with four legs',
    'a modern office chair with armrests',
    'a vintage armchair with floral pattern',
    'a minimalist stool made of metal',
    'a cozy bean bag chair',
]

for text in test_texts:
    print(f'\nGenerating: "{text}"')
    emb = torch.from_numpy(text_model.encode([text])).float()

    results = generate_and_visualize(
        gpt_model=gpt_model,
        vqvae=vqvae_model,
        text_emb=emb,
        device=device,
        temperature=0.8,
        top_k=100,
        top_p=0.95,
        num_samples=1,
    )
    print(f'Generated {len(results)} sample(s)')

In [ ]:
# ---- Save Final Models to Google Drive ----
import shutil

checkpoint_files = [
    'mae_best.pt',
    'vqvae_best.pt',
    'generator_best.pt',
]

for f in checkpoint_files:
    src = os.path.join('/content/drive/MyDrive/voxer_checkpoints', f)
    if os.path.exists(src):
        print(f'{f} saved: {os.path.getsize(src) / 1e6:.1f} MB')

print('\nAll stages complete!')